In [1]:
import re
import pandas as pd
import numpy as np

def converter_para_padrao_brasileiro(valor, manter_como_string=False):
    """
    Converte números para o padrão brasileiro (ponto para milhar, vírgula para decimal)

    Args:
        valor: Número ou string a ser convertido
        manter_como_string: Se True, retorna string formatada; se False, tenta retornar número

    Returns:
        Valor convertido para padrão brasileiro
    """
    if valor is None or pd.isna(valor):
        return valor if not manter_como_string else ''

    # Se já for número, converte para string no padrão brasileiro
    if isinstance(valor, (int, float, np.integer, np.floating)):
        if manter_como_string:
            # Formatar com separador de milhar (ponto) e decimal (vírgula)
            return f'{valor:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
        else:
            return valor

    # Converter para string se não for
    valor_str = str(valor).strip()

    # Se estiver vazio
    if not valor_str:
        return valor_str if manter_como_string else None

    # Remover caracteres não numéricos exceto ponto, vírgula e sinal negativo
    valor_limpo = re.sub(r'[^\d\-.,]', '', valor_str)

    # Detectar o formato atual e converter para padrão brasileiro
    # Caso 1: Já está no formato brasileiro (1.234.567,89)
    if re.match(r'^-?\d{1,3}(\.\d{3})*(,\d+)?$', valor_limpo):
        # Remover pontos dos milhares
        sem_pontos = valor_limpo.replace('.', '')
        # Substituir vírgula decimal por ponto (para operações numéricas)
        com_ponto = sem_pontos.replace(',', '.')

        if manter_como_string:
            # Manter formato brasileiro original
            return valor_limpo
        else:
            # Converter para número
            return float(com_ponto)

    # Caso 2: Formato americano (1,234,567.89)
    elif re.match(r'^-?\d{1,3}(,\d{3})*(\.\d+)?$', valor_limpo):
        # Remover vírgulas dos milhares
        sem_virgulas = valor_limpo.replace(',', '')

        if manter_como_string:
            # Converter para formato brasileiro
            partes = sem_virgulas.split('.')
            parte_inteira = partes[0]
            parte_decimal = partes[1] if len(partes) > 1 else ''

            # Adicionar pontos de milhar
            inteira_formatada = f'{int(parte_inteira):,}'.replace(',', '.')

            if parte_decimal:
                resultado = f'{inteira_formatada},{parte_decimal}'
            else:
                resultado = inteira_formatada

            return resultado
        else:
            # Converter para número
            return float(sem_virgulas)

    # Caso 3: Número simples (sem separadores)
    elif re.match(r'^-?\d+(\.\d+)?$', valor_limpo) or re.match(r'^-?\d+,\d+$', valor_limpo):
        # Verificar se usa vírgula decimal
        if ',' in valor_limpo and '.' not in valor_limpo:
            # Formato com vírgula decimal (sem milhares)
            com_ponto = valor_limpo.replace(',', '.')

            if manter_como_string:
                # Formatar com milhares se for grande
                num = float(com_ponto)
                return f'{num:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
            else:
                return float(com_ponto)
        else:
            # Formato com ponto decimal (sem milhares)
            if manter_como_string:
                num = float(valor_limpo)
                return f'{num:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
            else:
                return float(valor_limpo)

    # Caso 4: Não conseguiu identificar
    else:
        return valor_str if manter_como_string else valor_str


def tratar_dataframe_padrao_brasileiro(df, colunas=None, decimais=2, manter_como_string=False):
    """
    Aplica a conversão para padrão brasileiro em todas as colunas numéricas do DataFrame

    Args:
        df: DataFrame a ser convertido
        colunas: Lista específica de colunas para converter (se None, converte todas numéricas)
        decimais: Número de casas decimais para formatação
        manter_como_string: Se True, mantém como string formatada; se False, converte para número

    Returns:
        DataFrame com valores convertidos
    """
    df_resultado = df.copy()

    # Se não especificou colunas, tenta identificar colunas numéricas
    if colunas is None:
        colunas = df.select_dtypes(include=[np.number]).columns.tolist()

    for col in colunas:
        if col in df_resultado.columns:
            # Aplicar conversão
            df_resultado[col] = df_resultado[col].apply(
                lambda x: converter_para_padrao_brasileiro(x, manter_como_string)
            )

            # Se for manter como número, garantir tipo correto
            if not manter_como_string:
                df_resultado[col] = pd.to_numeric(df_resultado[col], errors='coerce')

    return df_resultado


def formatar_numero_brasileiro(valor, decimais=2, simbolo_moeda=False):
    """
    Formata um número para exibição no padrão brasileiro

    Args:
        valor: Número a ser formatado
        decimais: Número de casas decimais
        simbolo_moeda: Se True, adiciona R$ na frente

    Returns:
        String formatada (ex: 1.234,56)
    """
    if valor is None or pd.isna(valor):
        return ''

    try:
        num = float(valor)
        # Formatar com separadores
        formatado = f'{num:,.{decimais}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

        if simbolo_moeda:
            return f'R$ {formatado}'
        else:
            return formatado
    except (ValueError, TypeError):
        return str(valor)


def limpar_numero_brasileiro(valor_str):
    """
    Converte uma string no formato brasileiro (1.234,56) para float

    Args:
        valor_str: String no formato brasileiro

    Returns:
        Número float
    """
    if not isinstance(valor_str, str):
        return valor_str

    # Remover pontos dos milhares e substituir vírgula por ponto
    valor_limpo = valor_str.replace('.', '').replace(',', '.')

    # Remover tudo que não for número ou ponto (exceto sinal negativo)
    valor_limpo = re.sub(r'[^\d\-.]', '', valor_limpo)

    try:
        return float(valor_limpo)
    except ValueError:
        return np.nan


# Exemplos de uso e testes
if __name__ == "__main__":
    # Testes da função principal
    print("=" * 70)
    print("TESTES DE CONVERSÃO PARA PADRÃO BRASILEIRO")
    print("=" * 70)

    # Teste 1: Números americanos para brasileiro
    print("\n1. Convertendo formato americano -> brasileiro:")
    valores_americanos = ["1,234,567.89", "123.45", "1,000.00", "0.5"]
    for val in valores_americanos:
        convertido = converter_para_padrao_brasileiro(val, manter_como_string=True)
        print(f"  {val:15} -> {convertido}")

    # Teste 2: Números brasileiros para float
    print("\n2. Convertendo formato brasileiro -> número:")
    valores_brasileiros = ["1.234.567,89", "123,45", "1.000,00", "0,5"]
    for val in valores_brasileiros:
        convertido = converter_para_padrao_brasileiro(val, manter_como_string=False)
        print(f"  {val:15} -> {convertido}")

    # Teste 3: Formatação para exibição
    print("\n3. Formatando números para exibição (padrão brasileiro):")
    numeros = [1234567.89, 123.45, 1000.00, 0.5, 9876543.21]
    for num in numeros:
        formatado = formatar_numero_brasileiro(num, decimais=2)
        formatado_moeda = formatar_numero_brasileiro(num, decimais=2, simbolo_moeda=True)
        print(f"  {num:12.2f} -> {formatado:15} | {formatado_moeda}")

    # Teste 4: Limpeza de strings brasileiras
    print("\n4. Limpando strings brasileiras para números:")
    strings_br = ["R$ 1.234,56", "1.234.567,89", "Apenas 123,45 reais", "Valor: 1.000,00"]
    for val in strings_br:
        limpo = limpar_numero_brasileiro(val)
        print(f"  {val:30} -> {limpo}")

    # Teste 5: DataFrame com diferentes formatos
    print("\n5. Aplicando ao DataFrame:")
    df_teste = pd.DataFrame({
        'Preco': ["1.234,56", "2.345,67", "3.456,78"],
        'Volume': ["1,000,000", "2,500,000", "5,000,000"],
        'Retorno': [0.1234, 0.2345, 0.3456],
        'Percentual': ["12.34%", "23.45%", "34.56%"]
    })

    print("\nDataFrame original:")
    print(df_teste)

    # Converter colunas específicas
    df_convertido = tratar_dataframe_padrao_brasileiro(
        df_teste,
        colunas=['Preco', 'Volume'],
        manter_como_string=False
    )

    print("\nDataFrame após conversão (números):")
    print(df_convertido)
    print(f"\nTipos: {df_convertido.dtypes}")

    # Exibir formatado
    df_teste['Preco_Formatado'] = df_teste['Preco'].apply(
        lambda x: converter_para_padrao_brasileiro(x, manter_como_string=True)
    )
    print("\nDataFrame com formatação brasileira:")
    print(df_teste[['Preco', 'Preco_Formatado']])

    print("\n" + "=" * 70)
    print("✅ FUNÇÕES DE FORMATAÇÃO PRONTAS PARA USO")
    print("=" * 70)

TESTES DE CONVERSÃO PARA PADRÃO BRASILEIRO

1. Convertendo formato americano -> brasileiro:
  1,234,567.89    -> 1.234.567,89
  123.45          -> 123,45
  1,000.00        -> 1.000,00
  0.5             -> 0,5

2. Convertendo formato brasileiro -> número:
  1.234.567,89    -> 1234567.89
  123,45          -> 123.45
  1.000,00        -> 1000.0
  0,5             -> 0.5

3. Formatando números para exibição (padrão brasileiro):
    1234567.89 -> 1.234.567,89    | R$ 1.234.567,89
        123.45 -> 123,45          | R$ 123,45
       1000.00 -> 1.000,00        | R$ 1.000,00
          0.50 -> 0,50            | R$ 0,50
    9876543.21 -> 9.876.543,21    | R$ 9.876.543,21

4. Limpando strings brasileiras para números:
  R$ 1.234,56                    -> 1234.56
  1.234.567,89                   -> 1234567.89
  Apenas 123,45 reais            -> 123.45
  Valor: 1.000,00                -> 1000.0

5. Aplicando ao DataFrame:

DataFrame original:
      Preco     Volume  Retorno Percentual
0  1.234,56  1,0